# Preprocessing - MFCC Feature Extraction

Bu notebook veri setindeki ses dosyalar?ndan 3 saniyelik segmentler halinde MFCC ?zellikleri ??kar?r ve `dataset/features_3.0_sec.json` dosyas?n? ?retir.

In [ ]:
from pathlib import Path
import json
import math

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import scipy.fft

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock",
]


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "dataset" / "genres_original").exists():
            return candidate
    raise FileNotFoundError("Proje k?k? bulunamad?. Notebook'u music-genre-classification i?inde ?al??t?r?n.")

PROJECT_ROOT = find_project_root()
DATASET_DIR = PROJECT_ROOT / "dataset" / "genres_original"
FEATURES_PATH = PROJECT_ROOT / "dataset" / "features_3.0_sec.json"

print(f"Proje k?k?: {PROJECT_ROOT}")
print(f"Veri seti: {DATASET_DIR}")
print(f"??kt? JSON: {FEATURES_PATH}")

In [ ]:
for genre in GENRES:
    genre_dir = DATASET_DIR / genre
    wav_count = len(list(genre_dir.glob("*.wav"))) if genre_dir.exists() else 0
    print(f"T?r: {genre:10s} | Par?a adedi: {wav_count}")

missing = [genre for genre in GENRES if not (DATASET_DIR / genre).exists()]
if missing:
    raise FileNotFoundError(f"Eksik t?r klas?rleri: {missing}")

## ?rnek G?rselle?tirme

Bu b?l?m e?itim i?in zorunlu de?ildir; veri setinin do?ru okundu?unu g?stermek i?in birka? ?rnek ?izim ?retir.

In [ ]:
def waveform_plot(audio, sr, title):
    plt.figure(figsize=(12, 4))
    librosa.display.waveshow(audio, sr=sr)
    plt.title(title)
    plt.xlabel("Zaman (saniye)")
    plt.ylabel("Genlik")
    plt.tight_layout()


def freq_spectrum_plot(audio, sr, title):
    spec = scipy.fft.fft(x=audio)
    spec_db = 20 * np.log10(np.maximum(np.abs(spec), 1e-10))
    f_axis = np.linspace(0, sr, len(spec_db))
    half = len(spec_db) // 2

    plt.figure(figsize=(12, 4))
    plt.plot(f_axis[:half], spec_db[:half])
    plt.xscale("log")
    plt.xlim(20, sr / 2)
    plt.title(title)
    plt.xlabel("Frekans (Hz)")
    plt.ylabel("?iddet (dB)")
    plt.tight_layout()

sample_track = DATASET_DIR / "classical" / "classical.00000.wav"
audio_ex, sr_ex = librosa.load(sample_track, sr=22500)
waveform_plot(audio_ex, sr_ex, "Classical ?rnek waveform")
freq_spectrum_plot(audio_ex, sr_ex, "Classical ?rnek frekans spektrumu")

## MFCC ??karma

Model e?itiminde 30 saniyelik her par?a 10 par?aya b?l?n?r. B?ylece her ?rnek yakla??k 3 saniyelik bir segment olur.

In [ ]:
def extract_mfccs(
    dataset_dir: Path,
    output_path: Path,
    sample_rate: int = 22500,
    track_duration: int = 30,
    n_fft: int = 2048,
    hop_length: int = 512,
    n_mfcc: int = 13,
    num_segments: int = 10,
) -> dict:
    data = {
        "mapping": GENRES,
        "genre_name": [],
        "genre_num": [],
        "file": [],
        "segment": [],
        "mfcc": [],
    }

    samples_per_track = sample_rate * track_duration
    samples_per_segment = samples_per_track // num_segments
    expected_mfcc_vectors = math.ceil(samples_per_segment / hop_length)

    print("MFCC extraction ba?lad?")
    print("=======================")

    for genre_index, genre in enumerate(GENRES):
        genre_dir = dataset_dir / genre
        print(f"{genre.title()} i?leniyor...")

        for file_path in sorted(genre_dir.glob("*.wav")):
            try:
                signal, sr = librosa.load(file_path, sr=sample_rate, duration=track_duration)

                for segment_index in range(num_segments):
                    start_sample = segment_index * samples_per_segment
                    end_sample = start_sample + samples_per_segment
                    segment = signal[start_sample:end_sample]

                    if len(segment) < samples_per_segment:
                        segment = np.pad(segment, (0, samples_per_segment - len(segment)))

                    mfcc = librosa.feature.mfcc(
                        y=segment,
                        sr=sr,
                        n_fft=n_fft,
                        hop_length=hop_length,
                        n_mfcc=n_mfcc,
                    ).T

                    if len(mfcc) == expected_mfcc_vectors:
                        data["genre_name"].append(genre)
                        data["genre_num"].append(genre_index)
                        data["file"].append(str(file_path.relative_to(dataset_dir)))
                        data["segment"].append(segment_index)
                        data["mfcc"].append(mfcc.tolist())
            except Exception as exc:
                print(f"Atland?: {file_path.name} -> {exc}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as fp:
        json.dump(data, fp, indent=2)

    print("=======================")
    print(f"Kaydedildi: {output_path}")
    print(f"Toplam segment: {len(data['mfcc'])}")
    return data

In [ ]:
features = extract_mfccs(
    dataset_dir=DATASET_DIR,
    output_path=FEATURES_PATH,
    sample_rate=22500,
    track_duration=30,
    n_fft=2048,
    hop_length=512,
    n_mfcc=13,
    num_segments=10,
)

X_preview = np.array(features["mfcc"])
y_preview = np.array(features["genre_num"])
print(f"MFCC shape: {X_preview.shape}")
print(f"Label shape: {y_preview.shape}")